# Penyusunan Glosarium

Notebook ini hanya melakukan *post-processing* dari hasil ekstraksi per mata kuliah. Alurnya:

1. Baca `raw_<MK>.csv` tiap mata kuliah dari `data/raw/glosarium/<kode>/`.
2. (Opsional) terapkan keputusan manual lewat `OVERRIDE`.
3. Ambil baris `masuk`, urutkan abjad, beri `entri_id` (`PKK-001`, dst.), tambahkan `tanggal_kunci`.
4. Tulis CSV final (6 kolom), JSON, `frekuensi_materi`, dan `entri_isian` gabungan.

Cara mengubah keputusan: edit kolom `keputusan` (`masuk`/`tolak`) langsung di `raw_<MK>.csv`, atau tambahkan baris di sel *Override keputusan manual*. Lalu jalankan ulang semua sel. Batas 150 entri `masuk` per mata kuliah dicek di sel validasi.

In [1]:
import csv, json, re
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_DIR = ROOT / 'data' / 'raw' / 'glosarium'
OUT_DIR = ROOT / 'data' / 'processed' / 'glosarium'
MK_LIST = ['aok', 'aps', 'imk', 'jarnir', 'pkk', 'rpl']
TANGGAL_KUNCI = '2026-10-01'
CAP = 150
PEDOMAN = 'buku pedoman kurikulum 2024'

print('RAW_DIR :', RAW_DIR, RAW_DIR.exists())
print('OUT_DIR :', OUT_DIR)

RAW_DIR : c:\Users\Aufii\Documents\Informatika\OTHERS\8. RISET-SKRIPSI\assistive-communicative\data\raw\glosarium True
OUT_DIR : c:\Users\Aufii\Documents\Informatika\OTHERS\8. RISET-SKRIPSI\assistive-communicative\data\processed\glosarium


## 1. Baca data mentah

In [2]:
def read_csv(p):
    return pd.read_csv(p, dtype=str, keep_default_na=False, encoding='utf-8')

raw = {}
delivered = {}
for mk in MK_LIST:
    d = RAW_DIR / mk
    f = d / f'raw_{mk.upper()}.csv'
    assert f.exists(), f'tidak ditemukan: {f}'
    df = read_csv(f)
    df['mk'] = mk.upper()
    raw[mk] = df
    g = d / f'glosarium_{mk.upper()}.csv'
    delivered[mk] = read_csv(g) if g.exists() else None

pd.DataFrame({
    'mk': [m.upper() for m in MK_LIST],
    'baris_raw': [len(raw[m]) for m in MK_LIST],
    'masuk': [(raw[m].keputusan == 'masuk').sum() for m in MK_LIST],
    'tolak': [(raw[m].keputusan == 'tolak').sum() for m in MK_LIST],
    'ragu': [(raw[m].keputusan == 'ragu').sum() for m in MK_LIST],
})

,mk,baris_raw,masuk,tolak,ragu
0,AOK,369,150,219,0
1,APS,201,150,51,0
2,IMK,296,150,146,0
3,JARNIR,299,150,149,0
4,PKK,342,150,192,0
5,RPL,274,150,124,0


## 2. Override keputusan manual (opsional)

Cara termudah mengubah keputusan adalah mengedit kolom `keputusan` di `raw_<MK>.csv`. Sel ini untuk perubahan yang ingin dicatat di notebook saja, tanpa menyentuh file raw. Format: `(mk, kandidat, 'masuk'|'tolak', alasan)`. Biarkan kosong bila tidak perlu.

In [3]:
# (mk, kandidat, keputusan, alasan), contoh: ('rpl', 'refactoring', 'masuk', 'dinaikkan manual')
OVERRIDE = []
override_log = []
for mk, kand, kep, alasan in OVERRIDE:
    df = raw[mk]
    idx = df.index[df.kandidat == kand]
    assert len(idx) == 1, f'kandidat tidak ditemukan/tidak unik: {mk} {kand}'
    i = idx[0]
    override_log.append({'mk': mk.upper(), 'kandidat': kand, 'dari': df.at[i, 'keputusan'], 'menjadi': kep})
    df.at[i, 'keputusan'] = kep
    df.at[i, 'alasan'] = alasan
pd.DataFrame(override_log, columns=['mk', 'kandidat', 'dari', 'menjadi'])

,mk,kandidat,dari,menjadi


## 3. Validasi

In [ ]:
problems = []
for mk in MK_LIST:
    df = raw[mk]
    bad = set(df.keputusan) - {'masuk', 'tolak'}
    if bad: problems.append(f'{mk}: nilai keputusan tidak valid {bad}')
    m = df[df.keputusan == 'masuk']
    if len(m) > CAP: problems.append(f'{mk}: {len(m)} entri masuk, melebihi batas {CAP}')
    dup = m.kandidat[m.kandidat.str.casefold().duplicated()].tolist()
    if dup: problems.append(f'{mk}: kandidat ganda {dup}')
    miss = m[m.bentuk_permukaan_final.eq('') & m.bentuk_permukaan_padanan.eq('') & m.bentuk_permukaan_ditemukan.eq('')]
assert not problems, '\n'.join(problems)
print('validasi lolos')
pd.DataFrame({'mk': [m.upper() for m in MK_LIST], 'masuk': [(raw[m].keputusan == 'masuk').sum() for m in MK_LIST]})

validasi lolos


,mk,masuk
0,AOK,150
1,APS,150
2,IMK,150
3,JARNIR,150
4,PKK,150
5,RPL,150


## 4. Bangun entri final

- `entri_id`: entri `masuk` diurutkan abjad menurut `kandidat` (tanpa membedakan huruf besar/kecil), lalu diberi nomor `MK-001`, dst.
- `bentuk_permukaan`: kolom `bentuk_permukaan_final` dipecah menjadi satu bentuk per baris. Entri yang baru dinaikkan ke `masuk` dan belum punya bentuk final dibangun dari kanonik + padanan wajib + bentuk berfrekuensi ≥ 3.
- `sumber`: `materi ajar (file.pdf); buku pedoman kurikulum 2024` bila ada di materi dan pedoman, `materi ajar (file.pdf)` bila hanya di materi, `buku pedoman kurikulum 2024` untuk entri pedoman saja atau isian. Satu file saja: yang frekuensinya tertinggi. Untuk entri yang hanya terbaca dari gambar slide (frekuensi teks 0), file diambil dari `glosarium_<MK>.csv` hasil ekstraksi.

In [5]:
def parse_found(s):
    return {a.strip(): int(b) for a, b in re.findall(r'(.+?) \((\d+)\)(?: \| |$)', s)}

def top_file(per_file):
    best = None
    for part in [p.strip() for p in per_file.split(';') if p.strip()]:
        name, n = part.rsplit(':', 1)
        if best is None or int(n) > best[1]:
            best = (name.strip(), int(n))
    return best[0] if best else ''

def sumber_dari_lama(mk, entri_id_awal):
    g = delivered.get(mk)
    if g is None or not entri_id_awal: return ''
    s = g.loc[g.entri_id == entri_id_awal, 'sumber']
    return s.iloc[0] if len(s) else ''

def file_dari_sumber(s):
    m = re.match(r'materi ajar \((.*)\)(?:; buku pedoman kurikulum 2024)?$', s)
    return m.group(1) if m else ''

def bentuk_final(r):
    if r.bentuk_permukaan_final:
        return [x.strip() for x in r.bentuk_permukaan_final.split(' | ') if x.strip()]
    found = parse_found(r.bentuk_permukaan_ditemukan)
    out = [r.kandidat] + [x.strip() for x in r.bentuk_permukaan_padanan.split(' | ') if x.strip()]
    out += [f for f, n in found.items() if n >= 3]
    seen, res = set(), []
    for f in out:
        if f.casefold() not in seen:
            seen.add(f.casefold()); res.append(f)
    return res

entries = []      # satu baris per entri
for mk in MK_LIST:
    MK = mk.upper()
    m = raw[mk][raw[mk].keputusan == 'masuk'].copy()
    m = m.sort_values('kandidat', key=lambda s: s.str.casefold(), kind='stable')
    for n, (_, r) in enumerate(m.iterrows(), 1):
        eid = f'{MK}-{n:03d}'
        tf = top_file(r.frekuensi_per_file)
        lama = sumber_dari_lama(mk, r.entri_id_awal)
        if r.asal.startswith('buku pedoman') or r.asal == 'isian':
            sumber, file_utama = PEDOMAN, ''
        else:
            if not tf: tf = file_dari_sumber(lama)
            assert tf, f'file sumber tidak ditemukan untuk {MK} {r.kandidat}'
            sumber = f'materi ajar ({tf})' + (f'; {PEDOMAN}' if r.di_pedoman == 'ya' else '')
            file_utama = tf
        entries.append(dict(mata_kuliah=MK, entri_id=eid, bentuk_kanonik=r.kandidat, sumber=sumber,
                            bentuk=bentuk_final(r), found=parse_found(r.bentuk_permukaan_ditemukan),
                            total=int(r.frekuensi_total), file_utama=file_utama, asal=r.asal,
                            bahan_kajian=r.bahan_kajian, alasan=r.alasan, id_awal=r.entri_id_awal, di_pedoman=r.di_pedoman))
E = pd.DataFrame(entries)
print(len(E), 'entri')
E.groupby('mata_kuliah').size()

900 entri


mata_kuliah
AOK       150
APS       150
IMK       150
JARNIR    150
PKK       150
RPL       150
dtype: int64

## 5. Pemeriksaan: nomor entri dan isi dibanding hasil ekstraksi

Untuk entri yang tidak diubah, `entri_id` hasil renumber harus sama dengan `entri_id_awal` dari ekstraksi. Perbedaan hanya muncul bila kamu menambah override atau mengedit keputusan di raw.

In [6]:
E['id_sama'] = E.entri_id == E.id_awal
print(E.groupby('mata_kuliah').id_sama.agg(['sum', 'count']).rename(columns={'sum': 'id_sama', 'count': 'entri'}))

rows = [(e.mata_kuliah, f.casefold(), f, e.entri_id) for e in E.itertuples() for f in e.bentuk]
D = pd.DataFrame(rows, columns=['mk', 'kunci', 'bentuk', 'entri'])
g = D.groupby(['mk', 'kunci']).agg(n_entri=('entri', 'nunique'), bentuk=('bentuk', lambda x: ' / '.join(sorted(set(x)))),
                                   entri=('entri', lambda x: ', '.join(sorted(set(x)))), n=('bentuk', 'size')).reset_index()
antar_entri = g[g.n_entri > 1]
varian_huruf = g[(g.n_entri == 1) & (g.n > 1)]
print('bentuk yang sama muncul di lebih dari satu entri (dalam satu mata kuliah):', len(antar_entri))
print('varian huruf besar/kecil dalam entri yang sama (info):', len(varian_huruf))
assert antar_entri.empty, antar_entri
varian_huruf

             id_sama  entri
mata_kuliah                
AOK              150    150
APS              150    150
IMK              150    150
JARNIR           150    150
PKK              150    150
RPL              150    150
bentuk yang sama muncul di lebih dari satu entri (dalam satu mata kuliah): 0
varian huruf besar/kecil dalam entri yang sama (info): 2


,mk,kunci,n_entri,bentuk,entri,n
1024,JARNIR,internet of things,1,Internet of Things / internet of things,JARNIR-066,2
1195,JARNIR,zigbee,1,ZigBee / zigbee,JARNIR-150,2


## 6. Tulis keluaran

In [7]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / 'per_mk').mkdir(exist_ok=True)

glos = []
for e in E.itertuples():
    for f in e.bentuk:
        glos.append([e.mata_kuliah, e.entri_id, e.bentuk_kanonik, f, e.sumber, TANGGAL_KUNCI])
cols = ['mata_kuliah', 'entri_id', 'bentuk_kanonik', 'bentuk_permukaan', 'sumber', 'tanggal_kunci']
G = pd.DataFrame(glos, columns=cols)
G.to_csv(OUT_DIR / 'glosarium_semua.csv', index=False, encoding='utf-8')
for mk in MK_LIST:
    G[G.mata_kuliah == mk.upper()].to_csv(OUT_DIR / 'per_mk' / f'glosarium_{mk.upper()}.csv', index=False, encoding='utf-8')

js = {}
for e in E.itertuples():
    js.setdefault(e.mata_kuliah, {})[e.entri_id] = {
        'bentuk_kanonik': e.bentuk_kanonik, 'bentuk_permukaan': e.bentuk,
        'sumber': e.sumber, 'tanggal_kunci': TANGGAL_KUNCI}
with open(OUT_DIR / 'glosarium_semua.json', 'w', encoding='utf-8') as fh:
    json.dump(js, fh, ensure_ascii=False, indent=2)

fr = []
for e in E.itertuples():
    ff = e.file_utama
    for f in e.bentuk:
        fr.append([e.mata_kuliah, e.entri_id, e.bentuk_kanonik, f, e.found.get(f, 0), e.total, ff])
FR = pd.DataFrame(fr, columns=['mata_kuliah', 'entri_id', 'bentuk_kanonik', 'bentuk_permukaan',
                               'frekuensi_bentuk', 'frekuensi_entri', 'file_terbanyak'])
FR.to_csv(OUT_DIR / 'frekuensi_materi_semua.csv', index=False, encoding='utf-8')

jenis = {'isian': 'isian', 'buku pedoman (tidak ada di materi)': 'pedoman_saja', 'materi ajar (hanya di gambar slide)': 'gambar_materi'}
IS = E[E.asal.isin(jenis)].copy()
IS['jenis'] = IS.asal.map(jenis)
IS[['mata_kuliah', 'entri_id', 'bentuk_kanonik', 'jenis', 'bahan_kajian', 'alasan']].to_csv(
    OUT_DIR / 'entri_isian_semua.csv', index=False, encoding='utf-8')

pd.concat(raw.values()).to_csv(OUT_DIR / 'raw_semua_final.csv', index=False, encoding='utf-8')

ringkas = G.groupby('mata_kuliah').agg(entri=('entri_id', 'nunique'), baris=('entri_id', 'size'))
ringkas.loc['TOTAL'] = ringkas.sum()
ringkas

,entri,baris
mata_kuliah,,
AOK,150,297
APS,150,268
IMK,150,355
JARNIR,150,278
PKK,150,288
RPL,150,282
TOTAL,900,1768


## 7. Uji akhir

Memeriksa format CSV final, kunci unik, dan (untuk mata kuliah tanpa override) kesamaan isi dengan `glosarium_<MK>.csv` hasil ekstraksi. Setelah kamu mengedit raw, perbedaan dengan `glosarium_<MK>.csv` lama itu wajar.

In [8]:
assert list(G.columns) == cols
assert G.tanggal_kunci.eq(TANGGAL_KUNCI).all()
assert not G.duplicated(['mata_kuliah', 'entri_id', 'bentuk_permukaan']).any()
assert G.groupby('mata_kuliah').entri_id.nunique().max() <= CAP
assert G.sumber.str.match(r'^(materi ajar \(.+\)(; buku pedoman kurikulum 2024)?|buku pedoman kurikulum 2024)$').all()

diubah = {o[0].upper() for o in OVERRIDE}
for mk in MK_LIST:
    MK = mk.upper()
    lama = delivered[mk]
    if lama is None: continue
    baru = G[G.mata_kuliah == MK][cols]
    if MK in diubah:
        print(f'{MK}: diubah oleh override ({len(baru)} baris vs {len(lama)} hasil ekstraksi)')
    else:
        sama = set(map(tuple, baru.values)) == set(map(tuple, lama[cols].values))
        print(f'{MK}: identik dengan hasil ekstraksi = {sama}')
print('semua uji lolos')

AOK: identik dengan hasil ekstraksi = True
APS: identik dengan hasil ekstraksi = True
IMK: identik dengan hasil ekstraksi = True
JARNIR: identik dengan hasil ekstraksi = True
PKK: identik dengan hasil ekstraksi = True
RPL: identik dengan hasil ekstraksi = True
semua uji lolos
